# COMP5318 Assignment 1: Rice Classification

##### Group number: ...
##### Student 1 SID: ...
##### Student 2 SID: ...  
##### Student 3 SID: ... 
##### Student 4 SID: ... 

## **1. Data Pre-processing**

In [1]:
# Import all libraries
import numpy as np 
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  MinMaxScaler
from sklearn.model_selection import StratifiedKFold

In [2]:
# Ignore future warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [3]:
# Load the rice dataset: rice-final2.csv
df = pd.read_csv("rice-final2.csv", na_values="?")
df.isnull().sum()
df.shape
df.head()

,Area,Perimiter,Major_Axis_Length,Minor_Axis_Length,Eccentricity,Convex_Area,Extent,class
0,12573.0,461.466003,192.903351,84.572075,0.898772,12893.0,0.550433,class2
1,12845.0,464.121002,194.332214,85.524338,0.897952,13125.0,0.774962,class2
2,14055.0,488.748993,207.751755,87.250328,0.907536,14484.0,0.550076,class1
3,14412.0,490.324005,207.476135,89.689514,0.901735,14703.0,0.598853,class1
4,14658.0,477.117004,189.566635,99.997780,0.849551,15048.0,0.649504,class2


In [4]:
# Pre-process dataset

X = df.iloc[:,:-1]
y = df.iloc[:,-1]

# Fixing missing values by replacing the missing values with the mean of the column
imputer = SimpleImputer( missing_values = np.nan , strategy="mean")
X = imputer.fit_transform(X)

# Normalize between 0 and 1

scaler = MinMaxScaler()
X = scaler.fit_transform(X)

# Change Class 1 to 0 and Class 2 to 1
y = np.where(y=="class1" ,0, 1)

print(X.shape)
print(y.shape)


(1400, 7)
(1400,)


In [5]:
# Print first ten rows of pre-processed dataset to 4 decimal places as per assignment spec
# A function is provided to assist

def print_data(X, y, n_rows=10):
    """Takes a numpy data array and target and prints the first ten rows.
    
    Arguments:
        X: numpy array of shape (n_examples, n_features)
        y: numpy array of shape (n_examples)
        n_rows: numpy of rows to print
    """
    for example_num in range(n_rows):
        for feature in X[example_num]:
            print("{:.4f}".format(feature), end=",")

        if example_num == len(X)-1:
            print(y[example_num],end="")
        else:
            print(y[example_num])
            


In [6]:
print_data(X,y)

0.4628,0.5406,0.5113,0.4803,0.7380,0.4699,0.1196,1
0.4900,0.5547,0.5266,0.5018,0.7319,0.4926,0.8030,1
0.6109,0.6847,0.6707,0.5409,0.8032,0.6253,0.1185,0
0.6466,0.6930,0.6677,0.5961,0.7601,0.6467,0.2669,0
0.6712,0.6233,0.4755,0.8293,0.3721,0.6803,0.4211,1
0.2634,0.2932,0.2414,0.4127,0.5521,0.2752,0.2825,1
0.8175,0.9501,0.9515,0.5925,0.9245,0.8162,0.0000,0
0.3174,0.3588,0.3601,0.3908,0.6921,0.3261,0.8510,1
0.3130,0.3050,0.2150,0.5189,0.3974,0.3159,0.4570,1
0.5120,0.5237,0.4409,0.6235,0.5460,0.5111,0.3155,1


## **2. Build Classifiers**

- Part 1:  Logistic Regression, Naïve Bayes
- Part 2:  KNN, Decision Tree, Ada Boost, Gradient Boost, Random Forest, SVM

### Part 1: Cross-validation without parameter tuning

In [7]:
## Setting the 10 fold stratified cross-validation
cvKFold=StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

# The stratified folds from cvKFold should be provided to the classifiers

In [8]:
print(set(y))

{np.int64(0), np.int64(1)}


In [9]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

lr_model = LogisticRegression(random_state=0 , max_iter = 1000)

lr_scores = cross_val_score( lr_model, X, y , cv=cvKFold, scoring="accuracy")
lr_mean = lr_scores.mean()
print(lr_scores)
print("LogR average cross-validation accuracy: {:.4f}".format(lr_mean))

[0.91428571 0.93571429 0.96428571 0.94285714 0.95       0.92857143
 0.94285714 0.95       0.89285714 0.96428571]
LogR average cross-validation accuracy: 0.9386


In [10]:
# Naïve Bayes

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score
nb_model =GaussianNB()
nb_scores = cross_val_score(nb_model, X, y, cv= cvKFold, scoring= "accuracy")

nb_mean = nb_scores.mean()

print(nb_scores)
print("NB average cross-validation accuracy: {:.4f}".format(nb_mean))


[0.9        0.92857143 0.95714286 0.91428571 0.94285714 0.93571429
 0.94285714 0.94285714 0.86428571 0.93571429]
NB average cross-validation accuracy: 0.9264


### Part 1 Results


In [11]:
# Print results for each classifier in part 1 to 4 decimal places here:
print("LogR average cross-validation accuracy: {:.4f}".format(lr_mean))
print("NB average cross-validation accuracy: {:.4f}".format(nb_mean))

LogR average cross-validation accuracy: 0.9386
NB average cross-validation accuracy: 0.9264


## Reflection and Discussion for Part 1 

### Model Performance
Logistic Regression achieved an average accuracy of **0.9386**, while Naive Bayes achieved **0.9264**.

### Analysis
The better performance of Logistic Regression can be attributed to its ability to model relationships between features, whereas Naive Bayes assumes that all features are independent. In this dataset, features such as area and perimeter are likely correlated, which negatively impacts Naive Bayes.

### Model Comparison
- **Logistic Regression**: More flexible, captures feature relationships, higher accuracy  
- **Naive Bayes**: Simpler, faster, but assumes feature independence  

### Conclusion
Logistic Regression performed better overall, while Naive Bayes remains a computationally efficient alternative.

### Part 2: Cross-validation with parameter tuning

In [12]:
# KNN 
# parameters you may consider
k = [1, 3, 5, 7]
p = [1, 2]


In [13]:
# Decision Tree 
# parameters you may consider
max_depth = [3, 5, 7, 10]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]

In [14]:
# Ada Boost
# parameters you may consider
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

In [15]:
# Gradient Boost
# parameters you may consider
max_depth = [1, 3, 5, 7]
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

In [16]:
# Random Forest
# You should use RandomForestClassifier from sklearn.ensemble with information gain and max_features set to ‘sqrt’.
# parameters you may consider
n_estimators = [10, 30, 60, 100]
max_leaf_nodes = [6, 12]



In [17]:
# SVM
# parameters you may consider
C = [0.01, 0.1, 1, 5]
gamma = [0.01, 0.1, 1, 10]
# optional
kernel = []


### Part 2: Results

In [18]:
# Perform Grid Search with 10-fold stratified cross-validation (GridSearchCV in sklearn). 
# The stratified folds from cvKFold should be provided to GridSearchV

# This should include using train_test_split from sklearn.model_selection with stratification and random_state=0
# Print results for each classifier here. All the reported results should be printed to 4 decimal places except for the integers such as "k", "p", n_estimators" and "max_leaf_nodes".

# example printing:
print("KNN best k: ")
print("KNN best p: ")
print("KNN cross-validation accuracy: ")
print("KNN test set accuracy: ")

...

print("RF best n_estimators: ")
print("RF best max_leaf_nodes: ")
print("RF cross-validation accuracy: ")
print("RF test set accuracy: ")
print("RF test set macro average F1: ")
print("RF test set weighted average F1: ")

KNN best k: 
KNN best p: 
KNN cross-validation accuracy: 
KNN test set accuracy: 
RF best n_estimators: 
RF best max_leaf_nodes: 
RF cross-validation accuracy: 
RF test set accuracy: 
RF test set macro average F1: 
RF test set weighted average F1: 


### Test your code

In [19]:
#load the test dataset to test out your model 

test_df = pd.read_csv("test-before.csv" , na_values="?")

X = test_df.iloc[:,:-1]
y = test_df.iloc[:,-1]

# Fixing missing values by replacing the missing values with the mean of the column
imputer = SimpleImputer( missing_values = np.nan , strategy="mean")
X = imputer.fit_transform(X)

# Normalize between 0 and 1

scaler = MinMaxScaler()
X = scaler.fit_transform(X)

# Change Class 1 to 0 and Class 2 to 1
y = np.where(y=="class1" ,0, 1)

lr_scores_test = cross_val_score(lr_model, X, y, cv=cvKFold, scoring='accuracy')
nb_scores_test = cross_val_score(nb_model, X, y, cv=cvKFold, scoring='accuracy')

# print(lr_scores_test.mean())
# print(nb_scores_test.mean())




## **3. Reflection and Discussion**



## **AI Acknowledgement**